# 12 — Growth Opportunities

**Purpose:** Identify where the next wave of revenue will come from — category adoption and cross-sell.  
**Key question:** Which segments are underexposed to which categories, and what's the revenue potential?  

**Tables used:** `marts.mart_category_propensity`, `marts.mart_category_affinity`, `marts.mart_spend_momentum`

In [ ]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import os
PROJECT = os.environ.get('BQ_PROJECT', 'fmn-sandbox')
LOCATION = 'africa-south1'
client = bigquery.Client(project=PROJECT, location=LOCATION)

def q(sql):
    return client.query(sql).to_dataframe()

print(f'Connected to {PROJECT} ({LOCATION})')

## 1. Biggest revenue opportunities by segment + category
Where are the biggest gaps between current adoption and potential?

In [ ]:
top_opps = q(f"""
    SELECT segment_name, CATEGORY_TWO,
           adoption_rate_pct,
           unadopted_customers,
           ROUND(potential_revenue, 0) AS potential_revenue,
           propensity_level
    FROM `{PROJECT}.marts.mart_category_propensity`
    WHERE propensity_level IN ('Very High', 'High')
    ORDER BY potential_revenue DESC
    LIMIT 20
""")

if not top_opps.empty:
    print('Top 20 growth opportunities (high propensity segments x categories):')
    display(top_opps)
else:
    print('No propensity data. Run mart_category_propensity SQL first.')

## 2. Adoption heatmap
Which segments are over/under-indexed in each category?

In [ ]:
heatmap_data = q(f"""
    SELECT segment_name, CATEGORY_TWO, adoption_rate_pct
    FROM `{PROJECT}.marts.mart_category_propensity`
    WHERE CATEGORY_TWO IN (
        SELECT CATEGORY_TWO FROM `{PROJECT}.marts.mart_category_propensity`
        GROUP BY CATEGORY_TWO ORDER BY SUM(potential_revenue) DESC LIMIT 15
    )
""")

if not heatmap_data.empty:
    pivot = heatmap_data.pivot(index='segment_name', columns='CATEGORY_TWO', values='adoption_rate_pct')

    fig = px.imshow(pivot, text_auto='.0f', aspect='auto',
                    color_continuous_scale='YlGn',
                    title='Segment adoption rate by category (%)',
                    labels=dict(x='Category', y='Segment', color='Adoption %'))
    fig.update_layout(height=400)
    fig.show()

## 3. Revenue potential by segment
Which segment has the most untapped revenue across all categories?

In [ ]:
seg_potential = q(f"""
    SELECT segment_name,
           SUM(unadopted_customers) AS total_unadopted,
           ROUND(SUM(potential_revenue), 0) AS total_potential
    FROM `{PROJECT}.marts.mart_category_propensity`
    WHERE propensity_level IN ('Very High', 'High', 'Medium')
    GROUP BY segment_name
    ORDER BY total_potential DESC
""")

if not seg_potential.empty:
    fig = px.bar(seg_potential, x='segment_name', y='total_potential',
                 color='total_unadopted', color_continuous_scale='Blues',
                 title='Untapped revenue potential by segment',
                 labels={'total_potential': 'Potential revenue (R)', 'total_unadopted': 'Unadopted customers'})
    fig.show()
    display(seg_potential)

## 4. Cross-sell power pairs
Categories with highest affinity that customers havent crossed into yet.

In [ ]:
cross_sell = q(f"""
    SELECT category_a, category_b,
           shared_customers, lift,
           pct_a_also_shops_b, pct_b_also_shops_a,
           customers_in_a, customers_in_b
    FROM `{PROJECT}.marts.mart_category_affinity`
    WHERE lift > 1.5
      AND jaccard_pct < 25
    ORDER BY shared_customers DESC
    LIMIT 15
""")

if not cross_sell.empty:
    print('High-affinity, low-overlap pairs (best cross-sell targets):\n')
    for _, row in cross_sell.iterrows():
        gap_a = row['customers_in_a'] - row['shared_customers']
        gap_b = row['customers_in_b'] - row['shared_customers']
        print(f'  {row["category_a"]} + {row["category_b"]}')
        print(f'    lift: {row["lift"]:.1f}x | shared: {int(row["shared_customers"]):,}')
        print(f'    {int(gap_a):,} in A not yet in B | {int(gap_b):,} in B not yet in A')
        print()
else:
    print('No strong cross-sell pairs found.')

## 5. Accelerating customers — where to double down
Customers whose spend is growing. Which categories are they moving into?

In [ ]:
accel = q(f"""
    SELECT sm.momentum_status,
           COUNT(*) AS customers,
           ROUND(AVG(sm.spend_change_pct), 1) AS avg_growth,
           ROUND(AVG(sm.total_spend_12m), 0) AS avg_spend
    FROM `{PROJECT}.marts.mart_spend_momentum` sm
    WHERE sm.momentum_status = 'Accelerating'
    GROUP BY sm.momentum_status
""")

if not accel.empty:
    a = accel.iloc[0]
    print(f'Accelerating customers: {int(a["customers"]):,}')
    print(f'Average spend growth: +{a["avg_growth"]}%')
    print(f'Average 12-month spend: R{a["avg_spend"]:,.0f}')
    print(f'\nThese customers are increasing their engagement — ideal for upsell and new category introductions.')
else:
    print('No accelerating customers found.')

## 6. Summary: where to focus
Combining propensity, affinity, and momentum into actionable priorities.

In [ ]:
print('GROWTH STRATEGY SUMMARY\n')
print('1. CROSS-SELL: Target high-affinity category pairs with low overlap')
print('   - Customers in A but not B are the easiest to convert')
print('   - Use the lift scores to prioritise which pairs to target first\n')
print('2. CATEGORY EXPANSION: Push high-propensity segments into new categories')
print('   - Champions with 40% adoption in a category = 60% still untapped')
print('   - Focus on categories where avg spend per customer is highest\n')
print('3. MOMENTUM PLAYS: Double down on accelerating customers')
print('   - These customers are already growing organically')
print('   - Introducing them to new categories now has highest conversion potential\n')
print('4. DEFEND: Protect declining high-value customers')
print('   - Cross-reference with churn risk scores')
print('   - Retention campaign before they leave entirely')